# CIFAR-10 Transfer Learning

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchmetrics import Accuracy
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

## Load Data with ImageNet Normalization

In [2]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [3]:
train_dataset = datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Load Pretrained ResNet18

In [5]:
model = models.resnet18(weights=True)

/home/anilc/projects/pytorch-cifar/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [7]:
model.fc = nn.Linear(model.fc.in_features, 10)

## Training (Frozen Layers)

In [8]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(device)

cuda


In [11]:
num_epochs = 20
for epoch in range(1, num_epochs+1):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch: [{epoch}/{num_epochs}], Loss: {epoch_loss}")

Epoch: [1/20], Loss: 0.8408662291896313
Epoch: [2/20], Loss: 0.6187256121498239
Epoch: [3/20], Loss: 0.5924459312806654
Epoch: [4/20], Loss: 0.5704814520333429
Epoch: [5/20], Loss: 0.5694282056425538
Epoch: [6/20], Loss: 0.5599915431740948
Epoch: [7/20], Loss: 0.553972704102621
Epoch: [8/20], Loss: 0.5528986079766013
Epoch: [9/20], Loss: 0.5477077383024004
Epoch: [10/20], Loss: 0.5477094367489486
Epoch: [11/20], Loss: 0.5437548656369109
Epoch: [12/20], Loss: 0.5415453338409628
Epoch: [13/20], Loss: 0.5436969165644987
Epoch: [14/20], Loss: 0.5437725091834202
Epoch: [15/20], Loss: 0.5389502562982652
Epoch: [16/20], Loss: 0.5395820020409801
Epoch: [17/20], Loss: 0.5366054309146179
Epoch: [18/20], Loss: 0.5329906969042995
Epoch: [19/20], Loss: 0.535217327535
Epoch: [20/20], Loss: 0.5317715076000794


In [12]:
acc = Accuracy(task='multiclass', num_classes=10).to(device)

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        acc(outputs, labels)

    test_accuracy = acc.compute()
    print(f"Test Accuracy: {test_accuracy}")

Test Accuracy: 0.8087999820709229


## Data Augmentation Experiment

In [16]:
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

In [17]:
train_dataset = datasets.CIFAR10(root="./data", train=True, transform=train_transform, download=True)
test_dataset = datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [18]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)

/home/anilc/projects/pytorch-cifar/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(


In [19]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [20]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [22]:
num_epochs = 20
for epoch in range(1, num_epochs+1):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch: [{epoch}/{num_epochs}], Loss: {epoch_loss}")

Epoch: [1/20], Loss: 0.9813822789875137
Epoch: [2/20], Loss: 0.7456939550464415
Epoch: [3/20], Loss: 0.70456176199724
Epoch: [4/20], Loss: 0.6862226466236212
Epoch: [5/20], Loss: 0.6866072658306498
Epoch: [6/20], Loss: 0.6795269421604283
Epoch: [7/20], Loss: 0.6667167112193144
Epoch: [8/20], Loss: 0.6660035514008359
Epoch: [9/20], Loss: 0.6671087935833675
Epoch: [10/20], Loss: 0.6656319200992584
Epoch: [11/20], Loss: 0.6605424962156569
Epoch: [12/20], Loss: 0.6611041925142488
Epoch: [13/20], Loss: 0.6592065472813213
Epoch: [14/20], Loss: 0.6559177174821229
Epoch: [15/20], Loss: 0.6553924474341181
Epoch: [16/20], Loss: 0.6552661329202945
Epoch: [17/20], Loss: 0.6585824212530995
Epoch: [18/20], Loss: 0.6571047510332464
Epoch: [19/20], Loss: 0.6544308082755569
Epoch: [20/20], Loss: 0.652284259419612


In [23]:
acc = Accuracy(task='multiclass', num_classes=10).to(device)

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        acc(outputs, labels)

    test_accuracy = acc.compute()
    print(f"Test Accuracy: {test_accuracy}")

Test Accuracy: 0.8054999709129333


### Data Augmentation Results
Applied augmentation (flip, rotation, color jitter) but accuracy remained ~80%.

**Possible reasons:**
- Need more epochs (augmentation creates harder examples)
- CIFAR-10 low resolution (32×32) limits augmentation benefit

**Next steps for improvement:**
- Train longer (30-40 epochs)
- Fine-tune all layers instead of just fc